# 1. Librairies et paramètres

In [3]:
# année à traiter
year = "2026"

In [5]:
import os 
import pandas as pd
from pathlib import Path

# 2. Traitement du post process

Fonction pour le post traitement de la détection

In [4]:
def post_process_test(x):
    
    # ignorer les valeurs manquantes
    if pd.isna(x):
        return x
    x_str = str(int(float(x))) if x.replace('.', '', 1).isdigit() else str(x)

    # si exactement 4 chiffres et commence par "1" (dossard du 21 dont le 1er chiffre manque)
    if len(x_str) == 4 and x_str.startswith("1"):
        return "2" + x_str
    
    # si exactement 4 chiffres et commence par "0" (dossard du 10  dont le 1er chiffre manque)
    if len(x_str) == 4 and x_str.startswith("0"):
        return "1" + x_str

    return x_str

In [10]:
# Répertoire avec les résultats
results_repertory = Path('Results') / year
galeries = [d for d in os.listdir(results_repertory) if os.path.isdir(os.path.join(results_repertory, d))]

for galerie in galeries:

    # Lister les fichiers image dans le dossier
    galerie_path = Path(results_repertory) / galerie
    valids_extensions = ('.csv')
    files = sorted([f for f in os.listdir(galerie_path) if f.lower().endswith(valids_extensions)])

    # fichier à traiter
    file_name = files[0]
    file_path = Path(galerie_path) / file_name

    df_results = pd.read_csv(file_path)

    # 1er post-traitement (rajouter des chiffres s'il n'y en a que 4 détectés)
    df_results["text"] = df_results["text"].astype(str).apply(post_process_test)

    # 2ème post-traitement : regrouper les images 

    grouped = df_results.groupby("file")["text"].apply(list).reset_index()
    
    #max_dossards = grouped["text"].apply(len).max()
    max_dossards = 10
    
    # Création des colonnes "Dossard_1", "Dossard_2", ...
    for i in range(max_dossards):
        grouped[f"Dossard_{i+1}"] = grouped["text"].apply(lambda x: x[i] if i < len(x) else "")

    # On enlève la colonne "text" qui contient la liste brute
    df_final = grouped.drop(columns=["text"])

    # sauvegarde dans un nouveau fichier
    file_name_post_process = file_name[:-4] + "_post_process.csv"
    file_path_post_process = Path(galerie_path) / file_name_post_process
    df_final.to_csv(file_path_post_process, index=False, encoding='utf-8')
    print(f"Fichier enregistré : {file_path_post_process}")   


Fichier enregistré : Results\2026\Galerie_test\detection_post_process.csv


# 3. Concaténation de tous les fichiers

In [12]:
def concatenate_results_files():

    # Répertoire avec les résultats
    results_repertory = Path('Results') / year
    galeries = [d for d in os.listdir(results_repertory) if os.path.isdir(os.path.join(results_repertory, d))]

    all_dfs = []

    for galerie in galeries:

        # Lister les fichiers image dans le dossier
        galerie_path = Path(results_repertory) / galerie
        valids_extensions = ('post_process.csv')
        files = sorted([f for f in os.listdir(galerie_path) if f.lower().endswith(valids_extensions)])
        
        # Lecture du fichier
        file_name = files[0]
        file_path = Path(galerie_path) / file_name
        df_results = pd.read_csv(file_path)

        # ajout de la colonne Galerie
        df_results.insert(0, "Galerie", galerie_path)

        all_dfs.append(df_results)

        print(galerie_path, file_name)

    # concaténation des fichiers 
    if all_dfs: 
        df_concat = pd.concat(all_dfs, ignore_index=True)
        all_dfs_file_name = "all_results.csv"
        all_dfs_file_path = Path(results_repertory) / all_dfs_file_name
        df_concat.to_csv(all_dfs_file_path, index=False, encoding="utf-8")
    else:
        print("Pas de fichier à concaténer")

In [14]:
# appel de la fonction pour concaténer les fichiers de résultats
concatenate_results_files()

Results\2026\Galerie_test detection_post_process.csv
